In [1]:

# imports and load
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')

# Load the ML-ready dataset from Week 1
df = pd.read_csv('data/processed/telco_churn_features.csv')

print(f"Dataset loaded: {df.shape}")
print(f"Churn rate    : {df['Churn'].mean()*100:.1f}%")
print(f"\nAll columns:")
for col in df.columns:
    print(f"  {col}: {df[col].dtype}")

Dataset loaded: (7043, 24)
Churn rate    : 26.5%

All columns:
  gender: int64
  SeniorCitizen: int64
  Partner: int64
  Dependents: int64
  tenure: int64
  PhoneService: int64
  MultipleLines: int64
  InternetService: str
  OnlineSecurity: int64
  OnlineBackup: int64
  DeviceProtection: int64
  TechSupport: int64
  StreamingTV: int64
  StreamingMovies: int64
  Contract: str
  PaperlessBilling: int64
  PaymentMethod: str
  MonthlyCharges: float64
  TotalCharges: float64
  Churn: int64
  numServices: int64
  chargePerMonth: float64
  tenureBucket: str
  contractRisk: int64


In [2]:
# separate features (X) and target (y)

# Drop target column to get features
X = df.drop(columns=['Churn'])
y = df['Churn']

print(f"Features shape : {X.shape}")
print(f"Target shape   : {y.shape}")
print(f"\nTarget distribution:")
print(f"  Not churned (0): {(y==0).sum()} ({(y==0).mean()*100:.1f}%)")
print(f"  Churned     (1): {(y==1).sum()} ({(y==1).mean()*100:.1f}%)")

Features shape : (7043, 23)
Target shape   : (7043,)

Target distribution:
  Not churned (0): 5174 (73.5%)
  Churned     (1): 1869 (26.5%)


In [3]:
#  identify which columns need which preprocessing

# Numerical columns — need StandardScaler
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges',
                  'numServices', 'chargePerMonth', 'contractRisk']

# Categorical columns — need OneHotEncoder
categorical_cols = ['InternetService', 'Contract',
                    'PaymentMethod', 'tenureBucket']

# Binary columns — already 0/1, no preprocessing needed
binary_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
               'PhoneService', 'MultipleLines', 'OnlineSecurity',
               'OnlineBackup', 'DeviceProtection', 'TechSupport',
               'StreamingTV', 'StreamingMovies', 'PaperlessBilling']

print("Numerical cols  (StandardScaler)  :", numerical_cols)
print("\nCategorical cols (OneHotEncoder)  :", categorical_cols)
print("\nBinary cols      (passthrough)    :", binary_cols)

# Verify all columns are accounted for
all_cols = numerical_cols + categorical_cols + binary_cols
missing = [c for c in X.columns if c not in all_cols]
print(f"\nUnaccounted columns: {missing}")
print("(Should be empty list [])")

Numerical cols  (StandardScaler)  : ['tenure', 'MonthlyCharges', 'TotalCharges', 'numServices', 'chargePerMonth', 'contractRisk']

Categorical cols (OneHotEncoder)  : ['InternetService', 'Contract', 'PaymentMethod', 'tenureBucket']

Binary cols      (passthrough)    : ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling']

Unaccounted columns: []
(Should be empty list [])


In [4]:
#  split data 80/20 with stratification
# stratify=y ensures both splits have same churn rate (26.5%)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          
)

print("Train/Test Split Results:")
print(f"{'':20} {'Count':>8} {'Churn Rate':>12}")
print("-" * 42)
print(f"{'Training set':20} {len(X_train):>8} {y_train.mean()*100:>11.1f}%")
print(f"{'Test set':20} {len(X_test):>8}  {y_test.mean()*100:>11.1f}%")
print(f"{'Full dataset':20} {len(X):>8}  {y.mean()*100:>11.1f}%")
print("-" * 42)
print("\nChurn rates are equal in both splits — stratification worked!")

Train/Test Split Results:
                        Count   Churn Rate
------------------------------------------
Training set             5634        26.5%
Test set                 1409         26.5%
Full dataset             7043         26.5%
------------------------------------------

Churn rates are equal in both splits — stratification worked!


In [5]:
#  build sklearn ColumnTransformer
# This applies different preprocessing to different column types

# StandardScaler: normalises numerical cols to mean=0, std=1
# OneHotEncoder : converts text categories to binary columns
# passthrough    : binary cols passed as-is (already 0/1)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore',
                              sparse_output=False), categorical_cols),
        ('bin', 'passthrough', binary_cols)
    ]
)

print("Preprocessor built successfully!")
print("\nWhat it does:")
print("  StandardScaler  → normalises tenure, charges, numServices")
print("  OneHotEncoder   → expands InternetService, Contract, PaymentMethod, tenureBucket")
print("  passthrough     → keeps binary 0/1 columns unchanged")

Preprocessor built successfully!

What it does:
  StandardScaler  → normalises tenure, charges, numServices
  OneHotEncoder   → expands InternetService, Contract, PaymentMethod, tenureBucket
  passthrough     → keeps binary 0/1 columns unchanged


In [6]:
#  fit on training data only, transform both splits
# IMPORTANT: fit ONLY on X_train — never on X_test
# This prevents data leakage

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)   # only transform, not fit

print(f"X_train shape before preprocessing: {X_train.shape}")
print(f"X_train shape after preprocessing : {X_train_proc.shape}")
print(f"\nX_test shape before preprocessing : {X_test.shape}")
print(f"X_test shape after preprocessing  : {X_test_proc.shape}")
print(f"\nExtra columns added by OneHotEncoder:")
print(f"  {X_train_proc.shape[1] - X_train.shape[1]} new columns created")

X_train shape before preprocessing: (5634, 23)
X_train shape after preprocessing : (5634, 33)

X_test shape before preprocessing : (1409, 23)
X_test shape after preprocessing  : (1409, 33)

Extra columns added by OneHotEncoder:
  10 new columns created


In [7]:
#  get all feature names after preprocessing
# Useful for SHAP plots in Week 2 Day 5+

num_feature_names = numerical_cols
cat_feature_names = preprocessor.named_transformers_['cat']\
                                 .get_feature_names_out(categorical_cols).tolist()
bin_feature_names = binary_cols

all_feature_names = num_feature_names + cat_feature_names + bin_feature_names

print(f"Total features after preprocessing: {len(all_feature_names)}")
print("\nAll feature names:")
for i, name in enumerate(all_feature_names):
    print(f"  {i+1:2}. {name}")

Total features after preprocessing: 33

All feature names:
   1. tenure
   2. MonthlyCharges
   3. TotalCharges
   4. numServices
   5. chargePerMonth
   6. contractRisk
   7. InternetService_DSL
   8. InternetService_Fiber optic
   9. InternetService_No
  10. Contract_Month-to-month
  11. Contract_One year
  12. Contract_Two year
  13. PaymentMethod_Bank transfer (automatic)
  14. PaymentMethod_Credit card (automatic)
  15. PaymentMethod_Electronic check
  16. PaymentMethod_Mailed check
  17. tenureBucket_Growing
  18. tenureBucket_Loyal
  19. tenureBucket_Mature
  20. tenureBucket_New
  21. gender
  22. SeniorCitizen
  23. Partner
  24. Dependents
  25. PhoneService
  26. MultipleLines
  27. OnlineSecurity
  28. OnlineBackup
  29. DeviceProtection
  30. TechSupport
  31. StreamingTV
  32. StreamingMovies
  33. PaperlessBilling


In [9]:
# save everything needed for model training tomorrow
import joblib
import os

os.makedirs('models', exist_ok=True)

# Save preprocessor
joblib.dump(preprocessor, 'models/preprocessor.pkl')
print("Saved: models/preprocessor.pkl")

# Save processed arrays
np.save('models/X_train_proc.npy', X_train_proc)
np.save('models/X_test_proc.npy',  X_test_proc)
np.save('models/y_train.npy',      y_train.values)
np.save('models/y_test.npy',       y_test.values)
print("Saved: models/X_train_proc.npy")
print("Saved: models/X_test_proc.npy")
print("Saved: models/y_train.npy")
print("Saved: models/y_test.npy")

# Save feature names
import json
with open('models/feature_names.json', 'w') as f:
    json.dump(all_feature_names, f)
print("Saved: models/feature_names.json")

print("\nAll splits and preprocessor saved!")

Saved: models/preprocessor.pkl
Saved: models/X_train_proc.npy
Saved: models/X_test_proc.npy
Saved: models/y_train.npy
Saved: models/y_test.npy
Saved: models/feature_names.json

All splits and preprocessor saved!


In [12]:
# — day notes
notes = """
# Week 2 Day 1 — Train/Test Split + Preprocessing Pipeline

## What was done today

### Train/Test Split
- Split ratio : 80% train / 20% test
- Train size  : 5,634 customers
- Test size   : 1,409 customers
- stratify=y  : ensures equal churn rate in both splits (26.5%)
- random_state: 42 (reproducible results every run)

### Why stratify?
Without stratify, random chance could put more churners in train
than test, making evaluation unreliable. Stratification guarantees
the 26.5% churn rate is preserved in both splits.

### Preprocessing Pipeline (ColumnTransformer)
| Transformer    | Columns              | Why |
|----------------|----------------------|-----|
| StandardScaler | numerical (6 cols)   | XGBoost works better with normalised features |
| OneHotEncoder  | categorical (4 cols) | ML models need numbers not text |
| passthrough    | binary (13 cols)     | Already 0/1, no change needed |

### Data Leakage Prevention
- preprocessor.fit_transform() called ONLY on X_train
- preprocessor.transform() called on X_test (no fitting)
- This prevents test data statistics leaking into training

### Shape after preprocessing
- Before : 23 columns
- After  : 33 columns (OneHotEncoder added 10 binary columns)

## Files Saved
- models/preprocessor.pkl     ← reusable preprocessor
- models/X_train_proc.npy     ← processed training features
- models/X_test_proc.npy      ← processed test features
- models/y_train.npy          ← training labels
- models/y_test.npy           ← test labels
- models/feature_names.json   ← feature names for SHAP

## Tomorrow — Week 2 Day 2
- Train Logistic Regression as baseline model
- Evaluate: accuracy, F1, ROC-AUC, confusion matrix
- This gives us a benchmark to beat with XGBoost
"""

with open('reports/week2_notes.md', 'w' , encoding="utf-8") as f:
    f.write(notes)

print("Notes saved!")
print("\nWeek 2 Day 1 complete!")

Notes saved!

Week 2 Day 1 complete!
